In [51]:
# imports
import os
import csv
import pandas as pd

### Dataset A - Bryophyte Observations
*Contains all of the data we actually used for our bryophytes.*

We kept two files (in DatasetAData):
* `T01A_Site_Physical_Characteristics_2692088856810991747` (T01A) 
* `T19B_Moss_Identification__since_2009__11290156903414781363` (T19B) 

from: `publicABMIData7878735501232611176`.

### After preprocessing

T01A should contain:
* ABMI Site - joining key
* Year - joining key
* Public Latitude
* Public Longitude - for spatial filtering to parkland/North Saskatchewan/Red Deer region

T19B should contain:
* ABMI Site - joining key
* Year - joining key 
* Scientific Name - for counting species richness and filtering invalid records
* Taxonomic Resolution - filter to only count records identified to species level
* Common name - for ease of manual checks

### Preprocess_A():

Reads T01A (site coordinates) and T19B (moss identification) from
the DatasetAData folder, filters and merges them, then saves
annual aggregated features to preprocessing_results/.

Usage:
    python preprocess_A.py

VNA = Variable not applicable
SNI = Species not identified
SNR = Species needs review
UID = Unable to identify
DNC = Did not collect

In [52]:
# File paths
OUT_DIR    = "preprocessing_results"

# A
DATA_DIR_A   = "DatasetAData"

# B
DATA_DIR_B = "DatasetBData"

# C

# A
T01A_FILE  = "T01A_Site_Physical_Characteristics_2692088856810991747.csv"
T19B_FILE  = "T19B_Moss_Identification__since_2009__11290156903414781363.csv"

# B
YIELD_FILE = "3210035901-eng.csv"  

# A
# Invalid / missing-value codes used throughout ABMI raw data - from guide
ABMI_MISSING = {"SNI", "SNR", "DNC", "VNA", "NONE", "PNA", "UID", ""}

# B
# Statistics Canada missing value code
STATCAN_MISSING = {".."}

In [53]:
def preprocess_A():
    """
    Load, filter, merge, and aggregate the ABMI bryophyte dataset.

    Returns
    -------
    df_annual : pd.DataFrame
        One row per year with columns:
          Year, n_sites_sampled, species_richness,
          total_identifications, richness_per_site, ids_per_site
    """

    os.makedirs(OUT_DIR, exist_ok=True)

    # Load raw files
    t01a_raw = pd.read_csv(
        os.path.join(DATA_DIR_A, T01A_FILE),
        encoding="utf-8",
        low_memory=False,
        index_col=False,
    )
    t19b_raw = pd.read_csv(
        os.path.join(DATA_DIR_A, T19B_FILE),
        encoding="utf-8",
        low_memory=False,
        index_col=False,
    )

    print(f"  T01A raw shape : {t01a_raw.shape}")
    print(f"  T19B raw shape : {t19b_raw.shape}")

    # Strip whitespace from all column names (fixes trailing spaces in some headers)
    t01a_raw.columns = t01a_raw.columns.str.strip()
    t19b_raw.columns = t19b_raw.columns.str.strip()

    # We keep only the columns we need and drop duplicates for T01A
    t01a = (
        t01a_raw[["ABMI Site", "Year", "Public Latitude", "Public Longitude"]]
        .copy()
        .rename(columns={
            "ABMI Site"       : "abmi_site",
            "Year"            : "year",
            "Public Latitude" : "latitude",
            "Public Longitude": "longitude",
        })
    )

    # Convert to numeric - coerce bad values to NaN
    t01a["latitude"]  = pd.to_numeric(t01a["latitude"],  errors="coerce")
    t01a["longitude"] = pd.to_numeric(t01a["longitude"], errors="coerce")
    t01a["year"]      = pd.to_numeric(t01a["year"],      errors="coerce")

    # Drop missing coordinates or year
    t01a = t01a.dropna(subset=["latitude", "longitude", "year"])

    # One coordinate record per site-year
    t01a = t01a.drop_duplicates(subset=["abmi_site", "year"])

    print(f"  T01A after cleaning : {t01a.shape}")

    # We keep only the columns we need and drop invalid records for T19B
    t19b = (
        t19b_raw[["ABMI Site", "Year", "Scientific Name", "Taxonomic Resolution", "Common Name"]]
        .copy()
        .rename(columns={
            "ABMI Site"           : "abmi_site",
            "Year"                : "year",
            "Scientific Name"     : "scientific_name",
            "Taxonomic Resolution": "taxonomic_resolution",
            "Common Name"         : "common_name",
        })
    )

    t19b["year"] = pd.to_numeric(t19b["year"], errors="coerce")

    # Remove rows where scientific name is a missing value code
    before = len(t19b)
    t19b["scientific_name_clean"] = t19b["scientific_name"].astype(str).str.strip()
    t19b = t19b[~t19b["scientific_name_clean"].isin(ABMI_MISSING)]
    print(f"  T19B rows removed (invalid scientific name) : {before - len(t19b)}")

    # Keep only species-level identifications for consistent richness counts
    before = len(t19b)
    t19b = t19b[t19b["taxonomic_resolution"].astype(str).str.strip().str.lower() == "species"]
    print(f"  T19B rows removed (not species-level)       : {before - len(t19b)}")

    print(f"  T19B after cleaning : {t19b.shape}")

    # Merge coordinates onto T19B records
    t19b_merged = t19b.merge(t01a, on=["abmi_site", "year"], how="left")

    # Flag rows that couldn't be matched to a coordinate
    unmatched = t19b_merged["latitude"].isna().sum()
    if unmatched > 0:
        print(f"  WARNING: {unmatched} T19B rows have no matching T01A coordinate.")

    # CHECKPOINT - one example per unmatched site-year to understand why coordinates are missing
    unmatched_rows = t19b_merged[t19b_merged["latitude"].isna()]
    unmatched_unique = unmatched_rows.drop_duplicates(subset=["abmi_site", "year"])
    print(f"\n  CHECKPOINT - unique site-years dropped (no matching T01A coordinate):")
    print(unmatched_unique[["abmi_site", "year", "scientific_name_clean"]].to_string(index=False))

    # Drop unmatched rows
    t19b_merged = t19b_merged.dropna(subset=["latitude", "longitude"])

    # Save the merged cleaned file for inspection
    merged_path = os.path.join(OUT_DIR, "merged_bryophyte_cleaned.csv")
    t19b_merged.to_csv(merged_path, index=False)
    print(f"\n  Saved merged cleaned data -> {merged_path}")

    # Computes bioindicator features per year

    # Sites sampled each year (denominator for per-site normalization)
    sites_per_year = (
        t19b_merged.groupby("year")["abmi_site"]
        .nunique()
        .rename("n_sites_sampled")
    )

    # Species richness: unique species across all sites in a given year
    richness_per_year = (
        t19b_merged.groupby("year")["scientific_name_clean"]
        .nunique()
        .rename("species_richness")
    )

    # Total valid identifications (proxy for bryophyte abundance / detectability)
    ids_per_year = (
        t19b_merged.groupby("year")["scientific_name_clean"]
        .count()
        .rename("total_identifications")
    )

    df_annual = (
        pd.concat([sites_per_year, richness_per_year, ids_per_year], axis=1)
        .reset_index()
        .rename(columns={"year": "Year"})
        .sort_values("Year")
    )

    # Effort-normalized metrics
    df_annual["richness_per_site"] = (
        df_annual["species_richness"] / df_annual["n_sites_sampled"]
    ).round(4)

    df_annual["ids_per_site"] = (
        df_annual["total_identifications"] / df_annual["n_sites_sampled"]
    ).round(4)

    # drop years with fewer than 10 sites sampled

    year_site_counts = df_annual["n_sites_sampled"]
    usable_years     = df_annual[df_annual["n_sites_sampled"] >= 10]["Year"]

    before = len(df_annual)
    df_annual = df_annual[df_annual["Year"].isin(usable_years)]
    print(f"  Years removed (fewer than 10 sites) : {before - len(df_annual)}")
    print(f"  Usable years retained : {sorted(df_annual['Year'].tolist())}")

    # Save annual features
    annual_path = os.path.join(OUT_DIR, "bryophyte_annual_features.csv")
    df_annual.to_csv(annual_path, index=False)
    print(f"  Saved annual features     -> {annual_path}")
    print(f"\n  Annual features preview:\n{df_annual.to_string(index=False)}")

    return df_annual

In [ ]:
def preprocess_B():
    """
    Load, clean, and filter the Statistics Canada Alberta wheat yield dataset.

    Returns
    -------
    df_model : pd.DataFrame
        One row per year for 2010-2019 with columns:
          Year, wheat_yield_bu_acre
    """

    os.makedirs(OUT_DIR, exist_ok=True)

   # Load raw file - skip 8 metadata rows at top and footer content at bottom
    df_raw = pd.read_csv(
        os.path.join(DATA_DIR_B, YIELD_FILE),
        encoding="utf-8",
        skiprows=8,
        skipfooter=37,      # skips symbol legend, table corrections, footnotes, citation
        engine="python",    # required when using skipfooter
        on_bad_lines="skip",
    )

    print(f"  Raw shape : {df_raw.shape}")
    print(f"  Columns   : {list(df_raw.columns)}")
    print(f"  Raw sample:\n{df_raw.head(5).to_string()}")

    # Rename to standard column names - adjust left side if Stats Can header differs
    df = df_raw.rename(columns={
        df_raw.columns[0]: "Year",
        df_raw.columns[1]: "wheat_yield_bu_acre",
    })

    # Convert to numeric - coerce StatCan missing value ".." to NaN
    df["Year"]                = pd.to_numeric(df["Year"],                errors="coerce")
    df["wheat_yield_bu_acre"] = pd.to_numeric(df["wheat_yield_bu_acre"], errors="coerce")

    # Drop missing values
    before = len(df)
    df = df.dropna(subset=["Year", "wheat_yield_bu_acre"])
    print(f"  Rows dropped (missing or '..' values) : {before - len(df)}")

    df["Year"] = df["Year"].astype(int)

    # Save full historical series for EDA
    eda_path = os.path.join(OUT_DIR, "wheat_yield_full_historical.csv")
    df.to_csv(eda_path, index=False)
    print(f"\n  Saved full historical series -> {eda_path}  ({len(df)} years)")

    # Filter to modeling window 2010-2019
    df_model = df[(df["Year"] >= 2010) & (df["Year"] <= 2019)].copy()

    print(f"\n  Modeling window (2010-2019):\n{df_model.to_string(index=False)}")

    # Save modeling window
    model_path = os.path.join(OUT_DIR, "wheat_yield_2010_2019.csv")
    df_model.to_csv(model_path, index=False)
    print(f"\n  Saved modeling window -> {model_path}")

    return df_model

In [ ]:

if __name__ == "__main__":
    df_A = preprocess_A()
    print("\npreprocess_A complete.")

    df_B = preprocess_B()
    print("\npreprocess_B complete.")



  T01A raw shape : (5400, 14)
  T19B raw shape : (24422, 15)
  T01A after cleaning : (458, 4)
  T19B rows removed (invalid scientific name) : 5010
  T19B rows removed (not species-level)       : 5905
  T19B after cleaning : (13507, 6)

  CHECKPOINT - unique site-years dropped (no matching T01A coordinate):
 abmi_site  year      scientific_name_clean
      1083  2018        Ceratodon purpureus
      1085  2018 Eurhynchiastrum pulchellum
      1116  2018        Ceratodon purpureus
      1148  2018        Ceratodon purpureus
      1353  2018              Pohlia nutans
      1354  2018    Oncophorus wahlenbergii
      1378  2018  Ptilium crista-castrensis
      1402  2018         Lophozia longidens
      1482  2017            Bryum argenteum
      1355  2018       Climacium dendroides
      1379  2018         Tomentypnum nitens
      1404  2018  Campylophyllum hispidulum
      1482  2011            Tortula acaulon

  Saved merged cleaned data -> preprocessing_results\merged_bryophyte_clean

ParserError: Error tokenizing data. C error: Expected 2 fields in line 144, saw 4
